# The market half of a policy is worth 0 coins without its farm half

I spent about a week training an imitation policy on this competition and it never got past 2% of
a tape's score. I assumed I needed more data. I was wrong about what the problem was, and the
measurement that showed me is cheap enough that anyone going down the same road can run it first.

Here it is. Take two policies. Play the farm channel (`farmer` + `hands`) from one and the market
channel from the other, on the same town, against the same opponent.

| farm channel | market channel | coins |
|---|---|---|
| clone | clone | 2,423 |
| clone | **tape** | **0 — bankrupt** |
| **tape** | clone | 2,317 |
| tape | tape | **107,726** |

The tape's market orders, driven by a farm that did not grow what they assume, do not merely
underperform. They lose money until there is none: the policy sells stock it never harvested and
buys inputs the farm cannot use.

Neither channel carries the value. The value is in the two channels *agreeing* across 720 turns.

## Why this kills per-turn imitation learning

Supervised behaviour cloning optimises per-turn agreement with the teacher. That objective does
not preserve the thing above, and here is the direct evidence: turn-match accuracy is essentially
uncorrelated with money.

| policy | turns matching the teacher | coins |
|---|---|---|
| trained on a tape library | 33.7% | **3,061** |
| distilled from one strong tape | **76.4%** | 690 |
| the same, plus DAgger | 76.3% | 1,901 |
| the tape itself | — | ~160,000 |

The policy that agrees with its teacher on three quarters of turns earns a fifth of what the
policy agreeing on a third earns. If you are tuning your clone on validation accuracy, you are
tuning on a number that does not point at the score.

## The failure compounds, it does not average

The market head looked survivable offline. On held-out teacher states it still produced 64% of the
teacher's `BUY_SEED` orders, and it was 54.7% accurate on that class.

Rolling out on its own states, it issued **3 seed purchases in a season where the teacher issued
189** — 1.6%, not 64%.

That 40x collapse between offline and online is the whole story. A small early deficit puts the
farm somewhere the training distribution never went, and off-distribution the policy defaults to
the majority class, which in this game is *no order at all*. Mine over-predicted "no order" and
under-produced every action: sells −19%, seed −36%, land −62%.

Downstream, in a full season, the trace reads like a chain of dominoes:

| | clone | tape |
|---|---|---|
| BUY_SEED orders | 3 | 189 |
| BUY_LAND orders | 0 | 2 |
| tiles still locked at the end | 75 / 100 | 25 / 100 |
| WATER commands | 80 | 1,102 |
| HARVEST commands | ~0 | 476 |
| coins, weekly | 2,720 → 3,248 → 2,299 → 2,243 | 2,106 → 685 → 21,169 → 136,145 |

The low watering count is a *symptom*, not the bug. It never bought seed, so it never planted, so
there was nothing to water. And look at the coin row: the tape spends itself down to 685 in the
first week and comes back with 136,145. The clone never invests at all.

## Run the swap on your own policy

This is the check I wish I had run in the first hour. If your policy's two halves are separable —
if either half paired with a competent partner earns most of what the pair earns — then per-turn
imitation is a reasonable thing to spend a week on. If swapping a half costs you nearly everything,
that week will not come back.

It needs no training and no data beyond two policies and one season.

In [ ]:
def channel_swap(policy_a, policy_b, play_season):
    """Play all four combinations of two policies' farm and market channels.

    policy_a, policy_b : callables taking an observation and returning
                         {"farmer": [...], "hands": [...], "market": [...]}
    play_season        : callable taking one such policy and returning final coins,
                         with the town and the opponent held fixed across calls

    Returns coins for each combination. Read the two mixed rows: if either is close to the
    all-A or all-B row, the channels are separable and per-turn imitation can work. If both
    collapse, the policy is one plan and copying it turn by turn will not reproduce it.
    """
    def mixed(farm_from, market_from):
        def policy(obs):
            f = farm_from(obs) or {}
            m = market_from(obs) or {}
            return {"farmer": f.get("farmer") or ["PASS"],
                    "hands": f.get("hands") or [],
                    "market": m.get("market") or []}
        return play_season(policy)

    return {
        "farm A, market A": play_season(policy_a),
        "farm A, market B": mixed(policy_a, policy_b),
        "farm B, market A": mixed(policy_b, policy_a),
        "farm B, market B": play_season(policy_b),
    }


def separability(result):
    """Share of the better pure policy that survives the better mix. Mine was 2%."""
    pure = max(result["farm A, market A"], result["farm B, market B"])
    mix = max(result["farm A, market B"], result["farm B, market A"])
    return mix / pure if pure else float("nan")


# My numbers, for calibration:
mine = {"farm A, market A": 2423, "farm A, market B": 0,
        "farm B, market A": 2317, "farm B, market B": 107726}
print(f"separability: {separability(mine):.1%}")

## What I would do differently

I would not train a per-turn policy on this game again without running the swap first.

What the result argues for, if you want to build rather than copy, is optimising something that is
scored over a whole season — a hand-written policy whose structure keeps the two channels
consistent by construction, with its parameters searched against season-level outcome. Coupling
you get for free from the structure is coupling your learner does not have to discover.

I will also say what this does *not* show. It does not show that learning cannot work here. It
shows that a learner whose loss is per-turn agreement is optimising a quantity that is close to
orthogonal to the score, on this game, in my hands. If you have an imitation policy that beats its
teacher's tape, I would genuinely like to see it — it would mean the coupling is learnable from
supervision and I gave up too early.